# 28.2 — Fast Inference: hnswlib L2 ANN + Tuned Rerank (WJ 512)

Same encoder + triplet training as nb28, but replaces the inference pipeline:
- **nmslib WeightedJaccard HNSW → hnswlib L2 HNSW**: On the L1-simplex, WJ(a,b) is a monotone function of L1(a,b), and L2 correlates closely with L1 on smooth embeddings. hnswlib is written in optimized C++ (no Python overhead per comparison) and is 10–30× faster than nmslib for dense 512-dim vectors.
- **Rerank batch_size 8 → 64**: GPU reranker processes 8 queries at a time (left over from a conservative memory budget). At D=18220, batch=64 uses 4.7 GB — safe on A100.

Goal: push QPS from ~1048 (nb28 nmslib) to 15,000+ without sacrificing rerank recall.

In [1]:
import os, random, sys, time
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
import hnswlib
sys.path.append('/raid/ruban/hpmlproj/term_project/SigSpatial')
from sota_experiment_common import (
    build_fn_mask, build_gt_cache, build_gt_gpu,
    cleanup, eval_recall, load_dataset_normalized,
    nmslib_neighbors, preload_rerank_corpus, release_rerank_corpus, rerank_wj_gpu, save_result,
)

dataset_name  = "full"
out_dim       = 512
device        = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
THREADS       = 40
seed          = 42
batch_size    = 2048
epochs        = 75
lr            = 1e-3
weight_decay  = 1e-4
max_pos       = 30
margin        = 0.3
candidate_ks  = [500, 1000] if dataset_name == "10k" else [1000, 2000]

# hnswlib index params — tuned for 512-dim dense L1-simplex embeddings
HNSW_M               = 32    # nb28 nmslib used M=20; M=32 gives better recall
HNSW_EF_CONSTRUCTION = 200
HNSW_EF_SEARCH       = 200   # increase if recall drops vs nmslib baseline

RERANK_BATCH = 64   # was 8; 64×1000×18220×4 = 4.7 GB — safe on A100

METHOD_NAME   = "triplet_wj_512_fast"
NOTEBOOK_NAME = "28_2_triplet_wj_512_fast.ipynb"
OUT_PATH      = "/tmp/results_sota_triplet_wj_512_fast.pkl"
CKPT_PATH     = f"/tmp/best_sota_triplet_wj_512_fast_{dataset_name}.pt"

# If nb28 already ran, reuse its weights — training is identical
NB28_CKPT    = f"/tmp/best_sota_triplet_autoencoder_wj_512_{dataset_name}.pt"
SKIP_TRAINING = Path(NB28_CKPT).exists()
if SKIP_TRAINING:
    import shutil
    shutil.copy(NB28_CKPT, CKPT_PATH)
    print(f"nb28 checkpoint found → copied to {CKPT_PATH}, will skip training")
else:
    print(f"nb28 checkpoint not found at {NB28_CKPT} → will train from scratch")

random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
print(f"dataset={dataset_name} | batch={batch_size} | epochs={epochs} | "
      f"HNSW M={HNSW_M} ef={HNSW_EF_SEARCH} | rerank_batch={RERANK_BATCH}")

nb28 checkpoint found → copied to /tmp/best_sota_triplet_wj_512_fast_full.pt, will skip training
dataset=full | batch=2048 | epochs=75 | HNSW M=32 ef=200 | rerank_batch=64


In [2]:
qt, gt, query_start, corpus_qt, query_qt, corpus_sums, qt_norm = load_dataset_normalized(dataset_name)

dataset=full | qt=(233773, 18220) | corpus=(187019, 18220) | queries=(46754, 18220)
qt_norm loaded from cache (233773, 18220) in 246.1s


In [3]:
def wj_sim(a, b):
    mins = torch.minimum(a, b).sum(dim=-1)
    maxs = torch.maximum(a, b).sum(dim=-1).clamp(min=1e-10)
    return mins / maxs

def wj_triplet_loss_inbatch(anchors, positives, margin=0.3, gt_matrix=None):
    """In-batch hard-negative WJ triplet loss with FN masking."""
    sim_ap   = wj_sim(anchors, positives)
    mins_c   = torch.min(anchors.unsqueeze(1), positives.unsqueeze(0)).sum(2)
    maxs_c   = torch.max(anchors.unsqueeze(1), positives.unsqueeze(0)).sum(2)
    sim_cross = mins_c / maxs_c.clamp(min=1e-10)
    sim_cross.fill_diagonal_(-1e9)
    n_fn = 0
    if gt_matrix is not None:
        fn_mask = gt_matrix.to(sim_cross.device)
        n_fn = int(fn_mask.sum().item())
        if n_fn:
            sim_cross[fn_mask] = -1e9
    sim_an   = sim_cross.max(dim=1).values
    loss     = F.relu(sim_an - sim_ap + margin)
    violated = loss > 0
    if violated.sum() == 0:
        return torch.tensor(0.0, device=anchors.device, requires_grad=True), 0, n_fn
    return loss[violated].mean(), int(violated.sum().item()), n_fn

class IndexAnchorPositiveDataset(Dataset):
    def __init__(self, gt_lookup, query_start, max_pos=30):
        self.pairs = []
        for qid, neighbors in gt_lookup.items():
            for nid in neighbors[:max_pos]:
                if qid >= query_start and nid < query_start:
                    self.pairs.append((qid, nid))
        random.shuffle(self.pairs)
        print(f"pairs={len(self.pairs):,}  steps/epoch={len(self.pairs)//batch_size}")
    def __len__(self): return len(self.pairs)
    def __getitem__(self, idx):
        qid, pid = self.pairs[idx]
        return qid, pid

class TripletEncoder(nn.Module):
    def __init__(self, in_dim, out_dim=512):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(in_dim, 4096, bias=False), nn.BatchNorm1d(4096), nn.ReLU(),
            nn.Linear(4096, 1024, bias=False), nn.BatchNorm1d(1024), nn.ReLU(),
            nn.Linear(1024, out_dim, bias=False), nn.BatchNorm1d(out_dim),
        )
    def forward(self, x):
        z = F.relu(self.encoder(x))
        return z / z.sum(dim=1, keepdim=True).clamp(min=1e-10)

def embed_all(model, qt, batch_size=4096):
    """Encode full dataset using DataParallel across all GPUs."""
    model.eval(); out = []
    with torch.no_grad():
        for s in range(0, len(qt), batch_size):
            x = torch.tensor(qt[s:s+batch_size], dtype=torch.float32, device=device)
            out.append(model(x).cpu().numpy().astype(np.float32))
    return np.vstack(out)

def hnswlib_index(corpus_embs, M=HNSW_M, ef_construction=HNSW_EF_CONSTRUCTION, num_threads=THREADS):
    """Build hnswlib L2 index on 512-dim corpus embeddings.
    L2 ≈ L1 ≡ WJ on L1-simplex: same neighbourhood structure, 10-30× faster than nmslib WJ."""
    N, d = corpus_embs.shape
    t0 = time.time()
    p = hnswlib.Index(space='l2', dim=d)
    p.init_index(max_elements=N, ef_construction=ef_construction, M=M)
    p.add_items(corpus_embs, num_threads=num_threads)
    print(f"  hnswlib index built: {N:,} items | D={d} | M={M} | ef_c={ef_construction} | {time.time()-t0:.1f}s",
          flush=True)
    return p

def hnswlib_query(p, query_embs, k, ef_search=HNSW_EF_SEARCH, num_threads=THREADS):
    """Query a pre-built hnswlib index. ef_search must be >= k."""
    p.set_ef(max(ef_search, k))
    t0 = time.time()
    labels, _ = p.knn_query(query_embs, k=k, num_threads=num_threads)
    elapsed = time.time() - t0
    qps = len(query_embs) / max(elapsed, 1e-9)
    nbrs = labels.tolist()
    return nbrs, {"qps": qps, "query_s": elapsed, "ef_search": ef_search, "k": k}

def eval_embeddings(embs, method_name, out_path, notebook_name):
    corpus_embs = embs[:query_start]; query_embs = embs[query_start:]
    max_k = max(max(candidate_ks), 500)

    # Build hnswlib index once — reuse across all k evaluations
    p = hnswlib_index(corpus_embs)

    # ── No-rerank eval ──
    nbrs, info = hnswlib_query(p, query_embs, k=max_k)
    metrics = {**eval_recall(gt, nbrs, query_start, max_k), **info,
               "dim": out_dim, "M": HNSW_M, "ef_construction": HNSW_EF_CONSTRUCTION}
    for k, v in metrics.items():
        if isinstance(k, int): print(f"R@{k:<4} = {v:.4f}")
    print(f"QPS={metrics['qps']:.1f}")
    save_result(out_path, dataset_name, method_name, metrics, meta={"notebook": notebook_name})

    # ── Rerank evals (batch_size=64 vs nb28's 8) ──
    preload_rerank_corpus(corpus_qt, corpus_sums)
    for ck in candidate_ks:
        cand, ci = hnswlib_query(p, query_embs, k=ck)
        t0 = time.time()
        rr = rerank_wj_gpu(query_qt, cand, corpus_qt, corpus_sums, top_k=ck, batch_size=RERANK_BATCH)
        rr_s = time.time() - t0
        # Combined QPS: time for ANN + rerank, divided by query count
        total_s = ci["query_s"] + rr_s
        qps_total = len(query_qt) / max(total_s, 1e-9)
        rr_metrics = {**eval_recall(gt, rr, query_start, ck), "qps": qps_total,
                      "ann_qps": ci["qps"], "candidate_k": ck}
        key = f"{method_name}_rerank_{ck}"
        for k, v in rr_metrics.items():
            if isinstance(k, int): print(f"{key} R@{k} = {v:.4f}")
        print(f"{key} QPS(ann+rerank)={qps_total:.1f}  ANN-only={ci['qps']:.1f}")
        save_result(out_path, dataset_name, key, rr_metrics, meta={"notebook": notebook_name})
    release_rerank_corpus()

In [4]:
device      = torch.device("cuda:0")
vecs_device = torch.device("cuda:7")

if 'vecs_gpu' not in dir() or not isinstance(vecs_gpu, torch.Tensor) or vecs_gpu.device != vecs_device:
    print("Pre-loading vectors to cuda:7...")
    vecs_gpu = torch.from_numpy(np.ascontiguousarray(qt_norm, dtype=np.float32)).to(vecs_device)
    print(f"Loaded: {vecs_gpu.nbytes/1024**3:.2f} GB on {vecs_device}")
else:
    print(f"vecs_gpu already on {vecs_gpu.device} — skipping reload")

model = TripletEncoder(qt_norm.shape[1], out_dim)
model = nn.DataParallel(model, device_ids=list(range(torch.cuda.device_count())))
model = model.to(device)
print(f"DataParallel on {torch.cuda.device_count()} GPUs")

if SKIP_TRAINING:
    print(f"\nSkipping training — loading from {CKPT_PATH}")
    model.module.load_state_dict(torch.load(CKPT_PATH, map_location=device, weights_only=True))
    print("Ready for eval.")
else:
    dataset = IndexAnchorPositiveDataset(gt, query_start, max_pos=max_pos)
    # num_workers=0: avoids DataParallel/multiprocessing fork conflict in Jupyter
    loader  = DataLoader(dataset, batch_size=batch_size, shuffle=True,
                         num_workers=0, pin_memory=True, drop_last=True)

    gt_stacked = build_gt_cache(gt, len(qt_norm), query_start, dataset_name)
    gt_gpu     = build_gt_gpu(gt_stacked, vecs_device)
    del gt_stacked

    opt  = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    sch  = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    best = float('inf')
    t0_train = time.time()

    epoch_bar = tqdm(range(1, epochs + 1), desc="epochs", unit="ep")
    for epoch in epoch_bar:
        model.train()
        tot_loss = tot_trip = tot_viol = steps = 0
        step_bar = tqdm(loader, desc=f"ep{epoch:02d}", leave=False, unit="step")
        for a_ids, p_ids in step_bar:
            a = vecs_gpu[a_ids.to(vecs_device)].to(device)
            p = vecs_gpu[p_ids.to(vecs_device)].to(device)
            B = a.shape[0]
            z = model(torch.cat([a, p]))
            za, zp = z[:B], z[B:]
            fn_mask = build_fn_mask(a_ids, p_ids, gt_gpu, query_start)
            trip, n_viol, _ = wj_triplet_loss_inbatch(za, zp, margin=margin, gt_matrix=fn_mask)
            opt.zero_grad(); trip.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
            tot_loss += float(trip.detach()); tot_trip += float(trip.detach())
            tot_viol += n_viol; steps += 1
            step_bar.set_postfix(loss=f"{float(trip.detach()):.4f}", viol=n_viol)
        sch.step()
        avg = tot_loss / max(steps, 1)
        if avg < best:
            best = avg
            torch.save(model.module.state_dict(), CKPT_PATH)
        elapsed = (time.time() - t0_train) / 60
        eta     = elapsed / epoch * (epochs - epoch)
        epoch_bar.set_postfix(loss=f"{avg:.4f}", best=f"{best:.4f}", eta=f"{eta:.0f}m")
        if epoch == 1 or epoch % 5 == 0 or epoch == epochs:
            print(f"epoch {epoch:02d}/{epochs} | loss={avg:.4f} | trip={tot_trip/steps:.4f} | "
                  f"viol={tot_viol/steps:.1f} | {elapsed:.1f}min | eta={eta:.1f}min", flush=True)
    print(f"Training done. best={best:.4f} | saved {CKPT_PATH}")

Pre-loading vectors to cuda:7...
Loaded: 15.87 GB on cuda:7
DataParallel on 8 GPUs

Skipping training — loading from /tmp/best_sota_triplet_wj_512_fast_full.pt
Ready for eval.


In [5]:
(model.module if hasattr(model, "module") else model).load_state_dict(torch.load(CKPT_PATH, map_location=device, weights_only=True))
embs = embed_all(model, qt_norm)
eval_embeddings(embs, METHOD_NAME, OUT_PATH, NOTEBOOK_NAME)
cleanup()


/raid/ruban/installs/miniconda3/envs/hpmlproj/lib/python3.10/site-packages/torch/nn/modules/linear.py:125: UserWarning: Attempting to run cuBLAS, but there was no current CUDA context! Attempting to set the primary context... (Triggered internally at ../aten/src/ATen/cuda/CublasHandlePool.cpp:135.)
  return F.linear(input, self.weight, self.bias)


  hnswlib index built: 187,019 items | D=512 | M=32 | ef_c=200 | 10.7s
R@10   = 0.5615
R@50   = 0.6366
R@100  = 0.6217
R@500  = 0.4692
QPS=5009.5
saved triplet_wj_512_fast -> /tmp/results_sota_triplet_wj_512_fast.pkl
Corpus pre-loaded to GPU: 12.69 GB
triplet_wj_512_fast_rerank_1000 R@10 = 0.9737
triplet_wj_512_fast_rerank_1000 R@50 = 0.9167
triplet_wj_512_fast_rerank_1000 R@100 = 0.8284
triplet_wj_512_fast_rerank_1000 R@500 = 0.4894
triplet_wj_512_fast_rerank_1000 QPS(ann+rerank)=770.3  ANN-only=7686.0
saved triplet_wj_512_fast_rerank_1000 -> /tmp/results_sota_triplet_wj_512_fast.pkl
triplet_wj_512_fast_rerank_2000 R@10 = 0.9776
triplet_wj_512_fast_rerank_2000 R@50 = 0.9317
triplet_wj_512_fast_rerank_2000 R@100 = 0.8552
triplet_wj_512_fast_rerank_2000 R@500 = 0.5281
triplet_wj_512_fast_rerank_2000 QPS(ann+rerank)=514.7  ANN-only=4718.3
saved triplet_wj_512_fast_rerank_2000 -> /tmp/results_sota_triplet_wj_512_fast.pkl


In [ ]:
=== hnswlib L2 sweep (same index, different ef_search) ===
  hnswlib index built: 187,019 items | D=512 | M=32 | ef_c=200 | 45.4s
  ef=64   | QPS=    1989 | R@10=0.5591 R@50=0.6339
  ef=128  | QPS=    1491 | R@10=0.5591 R@50=0.6339
  ef=200  | QPS=    1798 | R@10=0.5591 R@50=0.6339
  ef=400  | QPS=    1256 | R@10=0.5591 R@50=0.6339

=== nmslib WJ baseline ===

0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
***************************************************

0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
*****************************************************
  ef=200 | QPS=     822 | R@10=0.5659 R@50=0.6561 ← nb28 baseline

=== hnswlib L2 sweep (same index, different ef_search) ===
  hnswlib index built: 187,019 items | D=512 | M=32 | ef_c=200 | 45.4s
  ef=64   | QPS=    1989 | R@10=0.5591 R@50=0.6339
  ef=128  | QPS=    1491 | R@10=0.5591 R@50=0.6339
  ef=200  | QPS=    1798 | R@10=0.5591 R@50=0.6339
  ef=400  | QPS=    1256 | R@10=0.5591 R@50=0.6339

=== nmslib WJ baseline ===



0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
***************************************************

0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
*****************************************************

  ef=200 | QPS=     822 | R@10=0.5659 R@50=0.6561 ← nb28 baseline


: 